<div style="border-left:8px solid #BC8F8F;background:#ffffff;padding:14px 18px;border-radius:4px"><div style="font:600 13px sans-serif;color:#BC8F8F;letter-spacing:.08em">PART A</div><div style="font:600 28px sans-serif;margin-top:2px;color:#1f2430">Presentation</div><div style="font:15px sans-serif;margin-top:6px;color:#1f2430">Watch and follow along. Searching a collection by what images look like.</div></div>

**Google Colab version.** Run the cell below first. It gets the workshop code, so that every path is the same as on the workshop server. Otherwise this notebook is the same.

In [ ]:
# Colab setup: run this cell first.
import os
import subprocess
import sys

if not os.path.exists("/content/DH-Workshop"):
    subprocess.run(["git", "clone", "-q", "https://github.com/laurajul/dh-workshop-dsi.git", "/content/DH-Workshop"], check=True)
os.chdir("/content/DH-Workshop/notebooks")
sys.path.insert(0, os.getcwd())
print("ready in", os.getcwd())

In [ ]:
%pip install -q ftfy regex git+https://github.com/openai/CLIP.git

## The shared data folder (Google Drive)

The collection (metadata and CLIP vectors) is in a shared Google Drive folder.

1. Run the first cell below and allow access to your Drive.
2. Open the folder with the link that the second cell shows. At the top, click the folder name, then **Organise, Add shortcut, My Drive**.
3. Run the second cell. It links the folder to `~/data`, the same path as on the workshop server.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import os
import ipywidgets as widgets

SHARED_FOLDER_URL = "PASTE-THE-LINK-TO-THE-SHARED-GOOGLE-DRIVE-FOLDER-HERE"     # <-- the instructor puts the link here
FOLDER_NAME = "DSI-Workshop"        # the name of the folder in your My Drive after adding the shortcut

display(widgets.HTML(f"<a href='{SHARED_FOLDER_URL}' target='_blank' style='font:16px sans-serif'>{SHARED_FOLDER_URL}</a>"))

src = f"/content/drive/MyDrive/{FOLDER_NAME}"
link = os.path.expanduser("~/data")
if os.path.islink(link) or os.path.exists(link):
    print(link, "is already there, leaving it as it is.")
elif not os.path.exists(os.path.join(src, "metadata.csv")):
    print("metadata.csv not found in", src, "- add the shortcut to My Drive (step 2) and check FOLDER_NAME.")
else:
    os.symlink(src, link)
    print("linked", src, "->", link)

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#BC8F8F"><span>PART A · 1 / 14</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #BC8F8F;margin:6px 0 0 0">

# Notebook 3: Searching by what images look like, and looking closer

The Europeana section of the intro notebook found objects through their *metadata*. This notebook searches by **content**: you describe an image in words, or give an image, and get the most similar ones from a shared collection.

It rests on an **embedding**: a list of numbers (here 512) that a neural network computes from an image, so that images with similar content get similar numbers. [CLIP](https://openai.com/research/clip) computes embeddings for images *and* short texts in the same space, so a sentence can be compared directly with every image.

In [ ]:
# Setup: run this cell first.
import numpy as np
from IPython.display import display
from clip_helpers import *
from region_helpers import *
from iiif_helpers import show_mirador_many, url_panel

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#BC8F8F"><span>PART A · 2 / 14</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #BC8F8F;margin:6px 0 0 0">

## How the dataset was made

The dataset contains **no images**, only numbers and links. It was prepared once by the instructor:

```
Europeana Search API  →  metadata records (title, creator, institution, rights, links)
        │   for every record: request a small version of the image
        │      from its IIIF image service, or from Europeana's thumbnail service
        ▼
CLIP image encoder  →  one vector of 512 numbers per image   (the image is then discarded)
```

Only openly reusable images from Europeana's **Art** collection are included. The **originals and the small versions stay at the institutions**: when a search returns results, your computer fetches *those* images from their URLs, and nothing else.

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#BC8F8F"><span>PART A · 3 / 14</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #BC8F8F;margin:6px 0 0 0">

## The collection

`metadata.csv` and the vectors are in the same order: row *i* describes vector *i*. A random sample (the images are fetched from their institutions now):

In [ ]:
coll = Collection()
coll.summary()

rng = np.random.default_rng(1)
coll.show_grid(rng.choice(len(coll.meta), 12, replace=False))

The Art collection is broader than paintings: it also holds drawings, prints, photographs, decorative objects, coins, maps and ethnographic items. Who contributes the most?

In [ ]:
coll.top_institutions_and_countries()

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#BC8F8F"><span>PART A · 4 / 14</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #BC8F8F;margin:6px 0 0 0">

## CLIP: text and images in one space

![CLIP contrastive pre-training: a text encoder and an image encoder produce vectors that are compared in a matrix](../misc/assets/img/clip.svg)

<small>Contrastive pre-training of CLIP: matching image-text pairs are pulled together (blue diagonal), all other pairs pushed apart. Figure: OpenAI, ["CLIP: Connecting text and images"](https://openai.com/index/clip/) (Radford et al., 2021).</small>

The image half was applied to the whole collection beforehand. For a query only the **text encoder** runs, which is fast on a laptop.

In [ ]:
clip_model = Clip()

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#BC8F8F"><span>PART A · 5 / 14</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #BC8F8F;margin:6px 0 0 0">

## Search with Europeana keywords

The Europeana search of the previous notebook looks for **words** in the metadata: title, description, creator, subject. Here a keyword search for *Amor and Psyche*, restricted to open images of the Art collection. Keep the result in mind: it is the comparison for the next section.

In [ ]:
import europeana_helpers as eu

KEYWORD = "Amor and Psyche"
result = eu.search_europeana(query=KEYWORD, rows=12, qf=["TYPE:IMAGE"], theme="art", reusability="open")
print(f"{result['totalResults']} records contain these words. The first 12:")
eu.show_results(result["items"])

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#BC8F8F"><span>PART A · 6 / 14</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #BC8F8F;margin:6px 0 0 0">

## Search with text

The query becomes a vector in the same 512-dimensional space. Both are normalised to length 1, so their **dot product** is their cosine similarity: 1 means the same direction, near 0 unrelated. Ranking the whole collection is one matrix–vector product.

In [ ]:
QUERY = "Amor and Psyche"

scores = coll.similarity(clip_model.text(QUERY))
best = coll.top_k(scores, 12)
coll.show_grid(best, scores)

**Keywords or vectors?** The keyword search is strict: every record names the pair, but a picture is only found if its catalogue entry says so. The vector search ranks by what the picture looks like and means. It finds some of the same works (*Cupid and Psyche*, *Psyche wird durch Amors Pfeil aus der Ohnmacht erweckt*) and a picture catalogued only as a *mythological scene*, but also other couples from myth and the Bible (Bacchus and Ariadne, Jupiter and Io, Adam and Eve). Neither is better: one answers *which records say this*, the other *which images look like this*.

The keyword search ran on all of Europeana's Art collection, the vector search runs on the shared sample of it.

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#BC8F8F"><span>PART A · 7 / 14</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #BC8F8F;margin:6px 0 0 0">

## Restricting the search with a mask

Because the metadata is aligned with the vectors, a filter is just a boolean **mask** applied before ranking. Here: only images dated before 1900.

In [ ]:
QUERY = "a portrait of a person"
mask = (coll.year > 0) & (coll.year < 1900)                          # dated, and before 1900
# mask = np.isin(coll.country, ["Sweden", "Finland", "Norway"])      # alternative: by country
# mask = np.isin(coll.institution, ["Wellcome Collection"])          # alternative: by institution

scores = coll.similarity(clip_model.text(QUERY))
print(f"{mask.sum():,} of {len(mask):,} images pass the filter. Top results for {QUERY!r}:\n")
coll.show_grid(coll.top_k(scores, 12, mask), scores)

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#BC8F8F"><span>PART A · 8 / 14</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #BC8F8F;margin:6px 0 0 0">

## Search with an image

The same space holds image embeddings, so an image can be the query: "more like this". **Upload an image** of your own (a photograph, a scan, a screenshot), then run the cell below it. Without an upload, the cell uses the best match of the text search above.

In [ ]:
import io
import ipywidgets as widgets
from PIL import Image

upload = widgets.FileUpload(accept="image/*", multiple=False, description="Upload an image")
display(upload)

In [ ]:
files = list(upload.value.values()) if isinstance(upload.value, dict) else list(upload.value)

if files:                                  # your own image
    query_image = Image.open(io.BytesIO(bytes(files[0]["content"]))).convert("RGB")
    print("Query: your image")
    display(query_image.copy().resize((200, int(200 * query_image.height / query_image.width))))
    scores = coll.similarity(clip_model.image(query_image))
    coll.show_grid(coll.top_k(scores, 12), scores)
else:                                      # no upload: more like the best match of the text search
    SEED = int(best[0])
    print("Query: collection image", SEED, "-", coll.meta[SEED]["title"][:60])
    display(coll.small_image(SEED))
    scores = coll.similarity(coll.V[SEED].astype(np.float32))
    scores[SEED] = -1                      # do not return the query itself
    coll.show_grid(coll.top_k(scores, 12), scores)

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#BC8F8F"><span>PART A · 9 / 14</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #BC8F8F;margin:6px 0 0 0">

## One picture, five regions

An image is rarely about one thing. We cut it into **regions**, embed each region on its own, and search the collection with it.

A Swedish advertising stamp (*reklammärke*) of 1939 for *Ljusäkta Fiona Färger*, from the Malmö Museum ([record](http://carlotta.malmo.se/carlotta-mmus/web/object/178916), CC BY 4.0). A kneeling figure by candlelight, a snake beside it, a band of coloured stripes below. We search with the whole stamp, the figure, only the face, one foot, the snake, and the motif without the perforated edge (the "teeth").

This record has no IIIF image service and its original file is not available, so the regions are cut from the small version (400 px) on your computer: coarse, which matters for small regions. `frac(x, y, w, h)` gives a region as fractions of the picture (0 to 1), converted to pixels.

The stamp is tall. With `fit="crop"` CLIP reads the central square, as it did for the vectors of the collection; the default `"pad"` would add grey bars and the search would follow the grey (try it).

In [ ]:
ID = "/91658/MM_objekt_178916"      # a record of the collection (Europeana ID)
row = next(i for i, r in enumerate(coll.meta) if r["id"] == ID)
pic = Picture(row, coll)

regions = {
    "kneeling figure":  pic.frac(0.20, 0.19, 0.50, 0.54),
    "face":             pic.frac(0.37, 0.20, 0.22, 0.17),
    "foot":             pic.frac(0.34, 0.59, 0.14, 0.13),
    "snake":            pic.frac(0.69, 0.54, 0.19, 0.20),
    "motif (no teeth)": pic.frac(0.055, 0.04, 0.87, 0.915),
}
pic.show(regions)

The same search with seven queries: the whole stamp and the five regions. Each row is one query, each card one of the 8 most similar images of the collection (the stamp itself is left out).

In [ ]:
exp = Experiment(coll, clip_model, pic)
exp.add("whole stamp", fit="crop")
for name, region in regions.items():
    exp.add(name, region, fit="crop")

exp.run(k=8)
exp.show()

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#BC8F8F"><span>PART A · 10 / 14</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #BC8F8F;margin:6px 0 0 0">

## Colour, grey, black and white

The IIIF **quality** is part of the URL, so the same region can be requested as `default`, `gray` or `bitonal` (pure black and white). Does the model still recognise it? A scan from a bitonal microfilm, or a photocopy, is such an image.

In [ ]:
q = Experiment(coll, clip_model, pic)
for quality in ("default", "gray", "bitonal"):
    q.add(f"whole, {quality}", quality=quality, fit="crop")
    q.add(f"face, {quality}", regions["face"], quality=quality, fit="crop")

q.run(k=8)
q.show()
q.overlap()

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#BC8F8F"><span>PART A · 11 / 14</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #BC8F8F;margin:6px 0 0 0">

## Regions against text

A text query lives in the same space, so it can be added as one more row: does the region find what the *words* find? Compare the snake region with the word "snake".

In [ ]:
exp.add("text: snake", text="a snake")
exp.add("text: reading", text="a person reading by candlelight")
exp.run(k=8)
exp.show()
exp.overlap()

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#BC8F8F"><span>PART A · 12 / 14</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #BC8F8F;margin:6px 0 0 0">

## Cut everything: tiles

Instead of choosing regions by hand, cut the whole picture into overlapping **tiles**, embed every tile, and search the tiles with a text. The result is a map of where in the picture something is. With IIIF each tile is one small request, so even a very large picture is possible (you will find large pictures in Part B).

In [ ]:
scan = TileScan(pic, clip_model, rows=4, overlap=0.5)
found = scan.show("a snake", k=3)

In [ ]:
other = scan.show("a person", k=3)

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#BC8F8F"><span>PART A · 13 / 14</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #BC8F8F;margin:6px 0 0 0">

## Annotations: pointing at regions

Every region becomes a **web annotation** that targets the canvas with `#xywh=`. The manifest also lists the images found for each region as canvases, grouped in one range per region. Open it in Mirador: the annotations are in the side panel, the images in the table of contents.

In [ ]:
notes = [
    note(regions["kneeling figure"],  "The kneeling figure. Dark painted figures and portraits: no stamp among the results.", hits=exp.top("kneeling figure", 3)),
    note(regions["face"],             "Only the face. Painted portraits of faces and children.", hits=exp.top("face", 3)),
    note(regions["snake"],            "The snake. The region finds a sabre and a signature, the word 'snake' finds snakes.", hits=exp.top("snake", 3)),
]
notes += [note(r, f"a snake? (tile score {s:.2f})") for r, s in found[:2]]

manifest = build_manifest("Demo: a stamp and its details", pic, notes, coll, summary="Regions of one advertising stamp and the images found for them.")
show_manifest(manifest)

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#BC8F8F"><span>PART A · 14 / 14</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #BC8F8F;margin:6px 0 0 0">

## The presentation

A `Deck` collects slides from your own results: text, the picture with its regions, the comparison, found images, and the annotated viewer. It is one HTML file (arrow keys to move).

In [ ]:
deck = Deck("Demo: a stamp and its details", author="Workshop", subtitle="Whole image vs. region")
deck.add_text("Question", "Does the collection answer a detail differently from the whole picture?")
deck.add_picture("Picture and regions", pic, regions)
deck.add_compare("Whole stamp vs. regions", exp, text="- The whole stamp finds its own series.\n- The motif without the teeth finds paintings of people reading by candlelight.\n- The snake and the foot are too small: arbitrary results.")
deck.add_mirador("Annotated", manifest)
deck.show()